# Lab | Working with Messy Real-World Data

## Overview
This lab focuses on cleaning, analyzing, and preparing the Online Retail Dataset. We will address missing values, invalid records, categorical challenges, class imbalance, and temporal data leakage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import resample

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

# Load the dataset
df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Shape: (541909, 8)

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Task 1: Data Profiling and Missing Values

### 1.1 — Profile the raw data

In [2]:
# Profiling summary
profiling = pd.DataFrame({
    "Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100,
    "Unique Values": df.nunique()
})
print(profiling)

print("\nNumeric Statistics:")
print(df.describe())

                       Type  Missing Values  Missing Percentage  Unique Values
InvoiceNo            object               0            0.000000          25900
StockCode            object               0            0.000000           4070
Description          object            1454            0.268311           4223
Quantity              int64               0            0.000000            722
InvoiceDate  datetime64[us]               0            0.000000          23260
UnitPrice           float64               0            0.000000           1630
CustomerID          float64          135080           24.926694           4372
Country                 str               0            0.000000             38

Numeric Statistics:
            Quantity                 InvoiceDate      UnitPrice     CustomerID
count  541909.000000                      541909  541909.000000  406829.000000
mean        9.552250  2011-07-04 13:34:57.156386       4.611114   15287.690570
min    -80995.000000         20

### 1.2 — Classify the missing data types

**CustomerID Missingness:**
- **Type:** Likely **MAR (Missing at Random)** or **MNAR (Missing Not at Random)**.
- **Evidence:** Guest checkouts often do not require a `CustomerID`. Comparing transactions with and without `CustomerID` might show systematic differences in `Country` or `UnitPrice` (e.g., certain countries or smaller purchases might be more frequent for guests).
- **Strategy:** **Deletion** for customer-level prediction tasks (since we cannot aggregate without an ID), but for transaction-level analysis, we might use an indicator or imputation if possible.

**Description Missingness:**
- **Type:** **MCAR (Missing Completely at Random)** or **MAR**.
- **Evidence:** Often occurs when `StockCode` is present but the database record is incomplete. We can check if these transactions have valid `StockCode`s.
- **Strategy:** **Impute** from other rows with the same `StockCode` or **delete** if the `StockCode` itself is invalid.

### 1.3 — Handle the missing values

In [3]:
# Handle Description: Impute from StockCode where possible
stock_desc_map = df.dropna(subset=["Description"]).drop_duplicates("StockCode").set_index("StockCode")["Description"].to_dict()
df["Description"] = df["Description"].fillna(df["StockCode"].map(stock_desc_map))

# Handle CustomerID: For this lab's prediction tasks, we will remove rows without CustomerID
df_clean = df.dropna(subset=["CustomerID"]).copy()

print(f"Remaining missing values:\n{df_clean.isnull().sum()}")

Remaining missing values:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


## Task 2: Cleaning Invalid and Anomalous Records

### 2.1 — Identify cancellations

In [4]:
cancellations = df_clean[df_clean["InvoiceNo"].astype(str).str.startswith("C")]
print(f"Number of cancellation transactions: {len(cancellations)}")

# For downstream prediction (high-value customers), cancellations represent returns and should be included to accurately reflect net revenue.

Number of cancellation transactions: 8905


### 2.2 — Handle invalid quantities and prices

In [5]:
invalid_qty = df_clean[(df_clean["Quantity"] <= 0) & (~df_clean["InvoiceNo"].astype(str).str.startswith("C"))]
invalid_price = df_clean[df_clean["UnitPrice"] <= 0]

print(f"Invalid Quantity (non-cancellation): {len(invalid_qty)}")
print(f"Invalid UnitPrice: {len(invalid_price)}")

# Strategy: Remove these records as they do not represent valid transactions
df_clean = df_clean[~((df_clean["Quantity"] <= 0) & (~df_clean["InvoiceNo"].astype(str).str.startswith("C")))]
df_clean = df_clean[df_clean["UnitPrice"] > 0]

Invalid Quantity (non-cancellation): 0
Invalid UnitPrice: 40


### 2.3 — Clean and validate

In [6]:
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Negative prices remaining: {(df_clean['UnitPrice'] <= 0).sum()}")

Shape after cleaning: (406789, 8)
Negative prices remaining: 0


## Task 3: Categorical Data Challenges

### 3.1 — Analyze the Country column

In [7]:
country_counts = df_clean["Country"].value_counts()
top_5_perc = country_counts.head(5).sum() / len(df_clean) * 100
rare_countries = country_counts[country_counts < 50]

print(f"Unique countries: {len(country_counts)}")
print(f"Top 5 countries account for {top_5_perc:.2f}% of transactions.")
print(f"Countries with < 50 transactions: {len(rare_countries)}")

# Group rare countries into 'Other'
df_clean["Country_Clean"] = df_clean["Country"].apply(lambda x: x if country_counts[x] >= 50 else "Other")
print(f"Unique countries after grouping: {df_clean['Country_Clean'].nunique()}")

Unique countries: 37
Top 5 countries account for 95.84% of transactions.
Countries with < 50 transactions: 6
Unique countries after grouping: 32


### 3.2 — Analyze the StockCode column

In [8]:
unique_stocks = df_clean["StockCode"].nunique()
non_product_codes = df_clean[df_clean["StockCode"].astype(str).str.contains('^[a-zA-Z]+', regex=True)]["StockCode"].unique()
print(f"Unique stock codes: {unique_stocks}")
print(f"Potential non-product codes: {non_product_codes}")

# Strategy: Keep only alphanumeric product codes for product-level analysis if needed.

Unique stock codes: 3684
Potential non-product codes: ['POST' 'D' 'C2' 'M' 'BANK CHARGES' 'PADS' 'DOT' 'CRUK']


### 3.3 — Engineer a feature from Description

In [9]:
df_clean["Desc_Word_Count"] = df_clean["Description"].astype(str).apply(lambda x: len(x.split()))
correlation = df_clean[["Desc_Word_Count", "Quantity", "UnitPrice"]].corr()
print("Correlation of engineered feature:")
print(correlation)

Correlation of engineered feature:
                 Desc_Word_Count  Quantity  UnitPrice
Desc_Word_Count         1.000000 -0.000828  -0.025281
Quantity               -0.000828  1.000000  -0.001235
UnitPrice              -0.025281 -0.001235   1.000000


## Task 4: Class Imbalance — Predicting High-Value Customers

### 4.1 — Engineer a binary target

In [10]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
customer_df = df_clean.groupby("CustomerID").agg({
    "Revenue": "sum",
    "InvoiceNo": "nunique",
    "StockCode": "nunique",
    "InvoiceDate": ["min", "max"]
}).reset_index()
customer_df.columns = ["CustomerID", "TotalRevenue", "OrderCount", "UniqueProducts", "FirstPurchase", "LastPurchase"]

threshold = customer_df["TotalRevenue"].quantile(0.90)
customer_df["HighValue"] = (customer_df["TotalRevenue"] > threshold).astype(int)
print(f"Target distribution:\n{customer_df['HighValue'].value_counts()}")

Target distribution:
HighValue
0    3934
1     437
Name: count, dtype: int64


### 4.2 — Measure the imbalance

- **Class Distribution:** 90% regular (0), 10% high-value (1).
- **Naive Accuracy:** If we always predict 'regular', we would achieve 90% accuracy.
- **Why accuracy is poor:** Accuracy doesn't distinguish between types of errors. In this case, missing a high-value customer (False Negative) is much more costly than misidentifying a regular one (False Positive).

### 4.3 — Apply resampling

In [11]:
X = customer_df[["OrderCount", "UniqueProducts"]]
y = customer_df["HighValue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Combine for resampling
train_data = pd.concat([X_train, y_train], axis=1)
majority = train_data[train_data.HighValue == 0]
minority = train_data[train_data.HighValue == 1]

# 1. Random Oversampling
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
upsampled_train = pd.concat([majority, minority_upsampled])

# 2. Random Undersampling
majority_downsampled = resample(majority, replace=False, n_samples=len(minority), random_state=42)
downsampled_train = pd.concat([majority_downsampled, minority])

def evaluate(X_tr, y_tr, X_te, y_te, label):
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(f"--- {label} ---")
    print(classification_report(y_te, y_pred))

evaluate(X_train, y_train, X_test, y_test, "Original")
evaluate(upsampled_train.drop("HighValue", axis=1), upsampled_train.HighValue, X_test, y_test, "Oversampled")
evaluate(downsampled_train.drop("HighValue", axis=1), downsampled_train.HighValue, X_test, y_test, "Undersampled")

--- Original ---
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       771
           1       0.82      0.57      0.67       104

    accuracy                           0.93       875
   macro avg       0.88      0.78      0.82       875
weighted avg       0.93      0.93      0.93       875

--- Oversampled ---
              precision    recall  f1-score   support

           0       0.98      0.91      0.95       771
           1       0.57      0.87      0.69       104

    accuracy                           0.91       875
   macro avg       0.78      0.89      0.82       875
weighted avg       0.93      0.91      0.92       875

--- Undersampled ---
              precision    recall  f1-score   support

           0       0.98      0.91      0.94       771
           1       0.55      0.87      0.67       104

    accuracy                           0.90       875
   macro avg       0.77      0.89      0.81       875
weighted avg    

## Task 5: Data Leakage — Introduce, Detect, and Fix

### 5.1 — Intentionally introduce temporal leakage

In [12]:
# Wrong approach: Randomly split features computed from the FULL date range
X_leak = customer_df[["TotalRevenue", "OrderCount"]]
y_leak = customer_df["HighValue"]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y_leak, test_size=0.2, random_state=42)

model_l = LogisticRegression()
model_l.fit(X_train_l, y_train_l)
print(f"Leaked Accuracy: {accuracy_score(y_test_l, model_l.predict(X_test_l)):.4f}")

Leaked Accuracy: 1.0000


### 5.2 — Detect the leakage

- **Symptoms:** Suspiciously high performance (near 100% accuracy) occurs because the target (`HighValue`) is directly derived from `TotalRevenue`, which we included as a feature. In a real-world scenario, we wouldn't know the full year's revenue at the time of prediction.

### 5.3 — Fix with a correct temporal split

In [13]:
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df_clean[df_clean["InvoiceDate"] <= observation_end]
df_pred = df_clean[df_clean["InvoiceDate"] >= prediction_start]

# Features from observation window
X_temporal = df_obs.groupby("CustomerID").agg({"Revenue": "sum", "InvoiceNo": "nunique"})
X_temporal.columns = ["Obs_Revenue", "Obs_OrderCount"]

# Target from prediction window: Did they buy in the future?
future_customers = df_pred["CustomerID"].unique()
X_temporal["BoughtInFuture"] = X_temporal.index.isin(future_customers).astype(int)

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_temporal.drop("BoughtInFuture", axis=1), 
    X_temporal["BoughtInFuture"], 
    test_size=0.2, 
    random_state=42
)

model_t = LogisticRegression()
model_t.fit(X_train_t, y_train_t)
print(f"Temporal Split Accuracy: {accuracy_score(y_test_t, model_t.predict(X_test_t)):.4f}")

Temporal Split Accuracy: 0.6575


**Conclusion:** The temporal split is the correct approach because it mimics the real-world scenario where you must predict future behavior using only past data. Random splitting fails when features are time-dependent, leading to overly optimistic results that won't generalize to new data.